# Aligned feature extraction — ~241 features

Consumes `01-alignment.ipynb` outputs and writes per-patient JSON + a merged CSV.

Categories: whole density, zone density (ETDRS + Wong), skeleton, caliber, Knudtson, color, tortuosity, FAZ, AV crossings, bifurcation, width variability, junction exponent.

## 0. Setup — imports and paths

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from features.zones import (
    make_etdrs_9subfields, make_disc_zones, make_standard_fov,
    mm_per_px_from_disc,
    ETDRS_9_NAMES, DISC_ZONE_NAMES,
    DEFAULT_ETDRS_RADII_MM, DEFAULT_DISC_MARGIN_ZONES,
    STANDARD_FOV_RADIUS_DD,
)
from features.density  import zone_density_features, whole_image_density_features
from features.skeleton import (
    whole_image_skeleton_features, whole_image_segment_features,
    whole_image_tortuosity_features, whole_image_bifurcation_features,
)
from features.caliber  import (
    whole_image_caliber_features, compute_crae_crve_avr,
    whole_image_width_variability, whole_image_junction_features,
)
from features.topology import whole_image_fractal_features, sholl_features
from features.color    import whole_image_color_features
from features.faz      import compute_faz
from features.crossings import crossings_features


In [2]:
SQUARE_SIZE  = 512
OUTPUT_ROOT  = PROJECT_ROOT / 'samples' / 'output'

rgb_path     = OUTPUT_ROOT / f'preprocessed_{SQUARE_SIZE}' / 'rgb'
aligned_root = OUTPUT_ROOT / f'aligned_{SQUARE_SIZE}'
features_out = OUTPUT_ROOT / f'features_json_aligned_{SQUARE_SIZE}'
features_out.mkdir(parents=True, exist_ok=True)

inv_df  = pd.read_csv(aligned_root / 'inventory.csv', parse_dates=['date'])
reg_log = pd.read_csv(aligned_root / 'registration_log.csv', parse_dates=['date'])

inventory = {}
for (pid, eye), grp in inv_df.groupby(['patient', 'eye'], sort=False):
    inventory.setdefault(pid, {})[eye] = grp.sort_values(['date', 'shot']).to_dict('records')

theta_cols = [f't{i}{j}' for i in range(3) for j in range(3)]
def get_theta(image_id):
    row = reg_log[reg_log['id'] == image_id]
    if len(row) == 0:
        return None
    return row.iloc[0][theta_cols].values.astype(np.float64).reshape(3, 3)

def warp_image(img, theta, H, W, is_mask=False):
    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    return cv2.warpAffine(img, theta[:2, :], (W, H),
                          flags=flags, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

patient_ids = sorted(inventory.keys())
for pid in patient_ids:
    for eye, visits in inventory[pid].items():
        print(f'{pid}/{eye}: {len(visits)} visits')

SAMPLE/R: 3 visits


## 1. Zones + common-valid mask per group

Zones built once from the fixed-visit geometry (same across visits). `common_valid` = fundus-pixel intersection across all visits of the group.

In [3]:
zones_per_group = {}
zone_meta = []
for pid, by_eye in inventory.items():
    for eye, visits in by_eye.items():
        if eye == 'unknown' or len(visits) < 1:
            continue
        fixed = visits[0]
        fx, fy = fixed['x_fovea'], fixed['y_fovea']
        dcx, dcy = fixed['x_disc'], fixed['y_disc']
        grp_df = inv_df[(inv_df['patient'] == pid) & (inv_df['eye'] == eye)]
        dd_px = grp_df['disc_diameter'].mean()
        mpp = mm_per_px_from_disc(dd_px)
        etdrs  = make_etdrs_9subfields(fx, fy, eye, mpp, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_ETDRS_RADII_MM)
        disc_z = make_disc_zones(dcx, dcy, dd_px, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_DISC_MARGIN_ZONES)
        zones_per_group[(pid, eye)] = {**etdrs, **disc_z}
        zone_meta.append({
            'patient': pid, 'eye': eye,
            'fovea_x': fx, 'fovea_y': fy,
            'disc_cx': dcx, 'disc_cy': dcy,
            'disc_diameter_px': dd_px, 'mm_per_px': mpp,
        })
zone_meta_df = pd.DataFrame(zone_meta)
zone_meta_df

,patient,eye,fovea_x,fovea_y,disc_cx,disc_cy,disc_diameter_px,mm_per_px
0,SAMPLE,R,360.5,265.299988,168.424698,237.091275,68.353206,0.026334


In [4]:
def fundus_valid(rgb):
    return rgb.sum(-1) > 10

def compute_common_valid(visits):
    common = np.ones((SQUARE_SIZE, SQUARE_SIZE), dtype=bool)
    for i, v in enumerate(visits):
        rgb = np.array(Image.open(rgb_path / f'{v["id"]}.png'))
        if i == 0:
            common &= fundus_valid(rgb)
        else:
            theta = get_theta(v['id'])
            if theta is None:
                continue
            common &= fundus_valid(warp_image(rgb, theta, SQUARE_SIZE, SQUARE_SIZE))
    return common

# common_valid = all-visit fundus intersection (zone / spatial features)
# whole_valid  = standard_fov (3*DD image-centered) & common_valid (whole-image features, fair across patients)
common_valid_per_group = {}
whole_valid_per_group  = {}
for key in zones_per_group.keys():
    pid, eye = key
    common = compute_common_valid(inventory[pid][eye])
    common_valid_per_group[key] = common

    dd_grp = zone_meta_df.query('patient == @pid & eye == @eye')['disc_diameter_px'].iloc[0]
    std_fov = make_standard_fov(
        cx=SQUARE_SIZE // 2, cy=SQUARE_SIZE // 2,
        disc_diameter_px=dd_grp, image_shape=(SQUARE_SIZE, SQUARE_SIZE),
        radius_dd=STANDARD_FOV_RADIUS_DD,
    )
    whole_valid_per_group[key] = std_fov & common

print(f'common_valid + whole_valid (3xDD image-ctr) computed for {len(common_valid_per_group)} group(s)')


common_valid computed for 1 group(s)


## 2. Feature extraction — all 12 categories in a single loop

In [5]:
feat_rows = []
for pid, by_eye in tqdm(inventory.items(), desc='patients'):
    for eye, visits in by_eye.items():
        if eye == 'unknown':
            continue
        key = (pid, eye)
        if key not in zones_per_group:
            continue
        zones        = zones_per_group[key]
        common_valid = common_valid_per_group[key]
        whole_valid  = whole_valid_per_group[key]
        meta         = zone_meta_df.query('patient == @pid & eye == @eye').iloc[0]
        mpp_grp      = meta['mm_per_px']
        dd_grp       = meta['disc_diameter_px']
        fx, fy       = meta['fovea_x'], meta['fovea_y']
        dcx, dcy     = meta['disc_cx'], meta['disc_cy']
        fixed_id     = visits[0]['id']
        aligned      = aligned_root / str(pid) / eye

        for v in visits:
            vid = v['id']
            try:
                rgb_base = np.array(Image.open(rgb_path / f'{vid}.png'))
                vs_m = np.array(Image.open(aligned / 'vessels' / f'{vid}.png'))
                av_m = np.array(Image.open(aligned / 'av'      / f'{vid}.png'))
                dc_m = np.array(Image.open(aligned / 'discs'   / f'{vid}.png'))
            except FileNotFoundError:
                continue

            dd_v = v.get('disc_diameter', np.nan)
            if np.isfinite(dd_v) and dd_v >= 5:
                mpp_v = mm_per_px_from_disc(dd_v)
            else:
                dd_v, mpp_v = float(dd_grp), float(mpp_grp)

            if vid == fixed_id:
                rgb_a = rgb_base
            else:
                theta = get_theta(vid)
                rgb_a = warp_image(rgb_base, theta, SQUARE_SIZE, SQUARE_SIZE) if theta is not None else rgb_base

            # Whole-image features (DD-scaled FOV: fair across patients)
            w_dens   = whole_image_density_features(vs_m, av_m, dc_m, valid_mask=whole_valid)
            w_skel   = whole_image_skeleton_features(vs_m, av_m, valid_mask=whole_valid)
            w_seg    = whole_image_segment_features(vs_m, av_m, valid_mask=whole_valid)
            w_cal    = whole_image_caliber_features(vs_m, av_m, valid_mask=whole_valid, mm_per_px=mpp_v)
            w_frac   = whole_image_fractal_features(vs_m, av_m, valid_mask=whole_valid)
            w_color  = whole_image_color_features(rgb_a, vs_m, av_m, valid_mask=whole_valid)
            w_tort   = whole_image_tortuosity_features(vs_m, av_m, valid_mask=whole_valid)
            w_bif    = whole_image_bifurcation_features(vs_m, av_m, valid_mask=whole_valid)
            w_wv     = whole_image_width_variability(vs_m, av_m, valid_mask=whole_valid)
            w_junc   = whole_image_junction_features(vs_m, av_m, valid_mask=whole_valid)

            # Zone / spatial features (common_valid only: DD-anchored zones, fovea/disc-anchored)
            z_dens   = zone_density_features(vs_m, av_m, dc_m, zones, valid_mask=common_valid)
            knudtson = compute_crae_crve_avr(av_m, zones['zone_B'], mm_per_px=mpp_v)
            w_sh_f   = sholl_features(vs_m, av_m, (fx, fy),   valid_mask=common_valid, center_name='fovea')
            w_sh_d   = sholl_features(vs_m, av_m, (dcx, dcy), valid_mask=common_valid, center_name='disc')
            faz      = compute_faz(vs_m, (fx, fy), valid_mask=common_valid, mm_per_px=mpp_v, disc_diameter_px=dd_v)
            cross    = crossings_features(av_m, zones=zones, valid_mask=common_valid)

            feat_rows.append({
                'id': vid, 'patient': pid, 'eye': eye,
                'date': v['date'], 'shot': v['shot'],
                'disc_diameter_px': dd_v, 'mm_per_px': mpp_v,
                **w_dens, **z_dens, **w_skel, **w_seg, **w_cal, **knudtson,
                **w_frac, **w_sh_f, **w_sh_d, **w_color,
                **w_tort, **faz, **cross, **w_bif, **w_wv, **w_junc,
            })

features_zones = pd.DataFrame(feat_rows).sort_values(['patient', 'eye', 'date']).reset_index(drop=True)
print(f'extracted features for {len(features_zones)} visits x {features_zones.shape[1]} columns')


patients: 100%|██████████| 1/1 [00:02<00:00,  2.73s/it]

extracted features for 3 visits × 248 columns


## 3. Save — per-patient JSON + merged CSV

In [6]:
def _jd(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        v = float(o)
        return None if not np.isfinite(v) else v
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, pd.Timestamp):
        return o.isoformat()
    return str(o)

n_saved = 0
for pid, pdf in features_zones.groupby('patient', sort=False):
    eye_meta = {}
    for eye in pdf['eye'].unique():
        mrow = zone_meta_df.query('patient == @pid & eye == @eye')
        if len(mrow) == 0:
            continue
        m = mrow.iloc[0]
        eye_meta[eye] = {
            'fovea_x': float(m['fovea_x']), 'fovea_y': float(m['fovea_y']),
            'disc_cx': float(m['disc_cx']), 'disc_cy': float(m['disc_cy']),
            'disc_diameter_px': float(m['disc_diameter_px']),
            'mm_per_px': float(m['mm_per_px']),
            'n_visits': int((pdf['eye'] == eye).sum()),
        }
    payload = {
        'patient': str(pid),
        'config': {
            'mode': 'aligned', 'square_size': SQUARE_SIZE,
            'extracted_at': datetime.now(timezone.utc).isoformat(),
        },
        'meta_by_eye': eye_meta,
        'features': pdf.assign(date=pdf['date'].astype(str)).to_dict('records'),
    }
    with open(features_out / f'{pid}.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=_jd)
    n_saved += 1

merged_csv = features_out / '_merged_features_aligned.csv'
features_zones.to_csv(merged_csv, index=False)
print(f'saved: {n_saved} patient JSON file(s) → {features_out}')
print(f'       + {merged_csv.name} (rows={len(features_zones)}, cols={features_zones.shape[1]})')
features_zones.head()

saved: 1 patient JSON file(s) → D:\fundus-vessel-features\samples\output\features_json_aligned_512
       + _merged_features_aligned.csv (rows=3, cols=248)


,id,patient,eye,date,shot,disc_diameter_px,mm_per_px,whole_vessel_density,whole_artery_density,whole_vein_density,...,whole_vein_width_cov_max,whole_vessel_junc_X_mean,whole_vessel_junc_X_std,whole_vessel_junc_n,whole_artery_junc_X_mean,whole_artery_junc_X_std,whole_artery_junc_n,whole_vein_junc_X_mean,whole_vein_junc_X_std,whole_vein_junc_n
0,pid_1,SAMPLE,R,2020-01-15,1,68.868115,0.026137,0.104145,0.047361,0.057977,...,0.327540,1.871233,0.537875,73,1.872727,0.699469,11,2.011765,0.699481,17
1,pid_2,SAMPLE,R,2021-01-20,1,67.768540,0.026561,0.098376,0.043010,0.053914,...,0.325196,1.741176,0.423478,68,1.903846,0.691281,13,1.680000,0.334440,20
2,pid_3,SAMPLE,R,2021-07-05,1,68.422963,0.026307,0.103163,0.046795,0.056459,...,0.423749,1.892647,0.759947,68,1.903571,0.364723,14,1.858824,0.580880,17


## 4. Verification — RAW vs WARPED side-by-side

For the first (patient, eye) group, compare the base-inference (RAW) mask against the
registered (WARPED) mask for FIXED visit, a middle visit, and the LAST visit.
This proves that feature extraction consumes the registered mask, not the raw one.

For each visit the figure shows:
- **Row 1 — RAW** (orange): original RGB + base-inference vessel mask + computed density.
   For non-fixed visits the raw mask sits in the per-visit coordinate frame, so the
   computed density does **not** match production (which uses WARPED).
- **Row 2 — WARPED** (blue): post-registration RGB (warped on the fly) + aligned vessel
   mask + computed density. This **matches** the production value stored in `features_zones`.

For FIXED visit (visit#0) both rows are identical because registration is the identity
transform. For later visits they diverge and the WARPED row is the one that matches.

Features verified: `whole_vessel_density`, `N2_vessel_density`, `zone_B_vessel_density`.


In [ ]:
import matplotlib.pyplot as plt
from features.density import region_density_features

# ─── Setup ──────────────────────────────────────────────
# base (pre-registration) vessel mask dir (same convention as 01-alignment)
base_vessels_path = OUTPUT_ROOT / f"vascx_output_{SQUARE_SIZE}" / "vessels"
assert base_vessels_path.exists(), f"base vessels dir not found: {base_vessels_path}"

pid, eye = next(iter(zones_per_group.keys()))
vlist         = inventory[pid][eye]
zones         = zones_per_group[(pid, eye)]
common_valid  = common_valid_per_group[(pid, eye)]
whole_valid   = whole_valid_per_group[(pid, eye)]
fixed_id      = vlist[0]["id"]

print(f"verifying eye {eye}  —  {len(vlist)} visits total")
print(f"→ RAW mask (base inference) vs WARPED mask (post-registration)")

def label(vi, v):
    tag = "FIXED" if vi == 0 else "warped"
    return f"visit#{vi} [{tag}]  {str(v['date'])[:10]}  eye={eye}  shot={v['shot']}"

def compare_raw_vs_warped(name, roi_mask, prod_value, vi, v):
    """2-row comparison: Row1=RAW (pre-registration), Row2=WARPED (production uses)."""
    vid = v["id"]
    is_fixed = (vid == fixed_id)

    # RAW: original RGB + base inference mask
    rgb_raw = np.array(Image.open(rgb_path / f"{vid}.png"))
    vs_raw  = np.array(Image.open(base_vessels_path / f"{vid}.png"))

    # WARPED: aligned mask + warped RGB (on-the-fly for non-fixed)
    vs_warp = np.array(Image.open(aligned_root / str(pid) / eye / "vessels" / f"{vid}.png"))
    if is_fixed:
        rgb_warp = rgb_raw
    else:
        theta = get_theta(vid)
        rgb_warp = warp_image(rgb_raw, theta, SQUARE_SIZE, SQUARE_SIZE) if theta is not None else rgb_raw

    def density(vs):
        v_in = (vs > 0) & roi_mask
        return int(v_in.sum()) / max(int(roi_mask.sum()), 1), int(v_in.sum())

    dens_raw,  vin_raw  = density(vs_raw)
    dens_warp, vin_warp = density(vs_warp)
    roi_n = int(roi_mask.sum())

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))

    for row, (which, rgb, vs, vin, dens, color_tag) in enumerate([
        ("RAW  (pre-registration — base inference)",     rgb_raw,  vs_raw,  vin_raw,  dens_raw,  "orange"),
        ("WARPED  (post-registration — production uses)", rgb_warp, vs_warp, vin_warp, dens_warp, "blue"),
    ]):
        # Col 0: RGB
        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"{which}\nRGB", fontsize=10, color=color_tag)
        axes[row, 0].axis("off")

        # Col 1: mask + ROI overlay
        disp = rgb.copy()
        disp[~roi_mask] = (disp[~roi_mask] * 0.4).astype(np.uint8)
        v_bin = vs > 0
        v_in  = v_bin & roi_mask
        v_out = v_bin & ~roi_mask
        overlay = disp.astype(np.float32)
        overlay[v_out] = 0.5 * overlay[v_out] + 0.5 * np.array([180, 180, 180])
        overlay[v_in]  = 0.3 * overlay[v_in]  + 0.7 * np.array([230,  30,  30])
        axes[row, 1].imshow(overlay.clip(0, 255).astype(np.uint8))
        axes[row, 1].set_title(f"ROI = {name}\n"
                                f"vessel ∩ ROI (red) = {vin:,} px  /  ROI = {roi_n:,} px",
                                fontsize=10)
        axes[row, 1].axis("off")

        # Col 2: numeric comparison
        axes[row, 2].axis("off")
        match = np.isclose(dens, prod_value, atol=1e-6)
        if row == 0:
            if is_fixed:
                verdict = "— identical to WARPED (fixed visit, no warping)"
                vcolor  = "gray"
            else:
                verdict = ("✗ RAW does NOT match production\n"
                           "   → raw mask is in the original frame\n"
                           "   → production uses WARPED (next row)")
                vcolor  = "red"
        else:
            verdict = "✓ MATCH production" if match else "✗ MISMATCH"
            vcolor  = "green" if match else "red"

        text = (f"density  = {vin:,} / {roi_n:,}\n"
                f"         = {dens:.6f}\n"
                f"{'─'*32}\n"
                f"production : {prod_value:.6f}\n"
                f"delta      : {dens - prod_value:+.2e}\n"
                f"{'─'*32}\n"
                f"{verdict}")
        axes[row, 2].text(0.02, 0.95, text, family="monospace", fontsize=11,
                 va="top", ha="left", color=vcolor,
                 transform=axes[row, 2].transAxes)

    fig.suptitle(f"VERIFY  {name}   —   {label(vi, v)}",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

# ─── Pick N evenly-spaced visits across all visits (covers full drift range) ───
N_SHOW = 5
if len(vlist) <= N_SHOW:
    show_indices = list(range(len(vlist)))
else:
    show_indices = np.linspace(0, len(vlist) - 1, N_SHOW).astype(int).tolist()
show_indices = sorted(set(show_indices))

print(f"\n═══ whole_vessel_density: visits {show_indices} out of {len(vlist)} ═══")
for vi in show_indices:
    v = vlist[vi]; vid = v["id"]
    prod_row = features_zones[(features_zones["patient"] == pid) &
                               (features_zones["eye"] == eye) &
                               (features_zones["id"] == vid)].iloc[0]
    compare_raw_vs_warped("whole_vessel_density", whole_valid,
                           prod_row["whole_vessel_density"], vi, v)

# ─── N2 + zone_B for FIXED and LAST ───
nt_indices = [0]
if len(vlist) >= 2: nt_indices.append(len(vlist) - 1)
print(f"\n═══ N2 + zone_B: visits {nt_indices} ═══")
for vi in nt_indices:
    v = vlist[vi]; vid = v["id"]
    prod_row = features_zones[(features_zones["patient"] == pid) &
                               (features_zones["eye"] == eye) &
                               (features_zones["id"] == vid)].iloc[0]
    compare_raw_vs_warped("N2_vessel_density",     zones["N2"]     & common_valid,
                           prod_row["N2_vessel_density"], vi, v)
    compare_raw_vs_warped("zone_B_vessel_density", zones["zone_B"] & common_valid,
                           prod_row["zone_B_vessel_density"], vi, v)


In [ ]:
# ─── 4-4. Batch verification — all density features for this visit ───
print("─" * 70)
print(f"Batch recompute vs production output for all density features")
print(f"  sample: {pid}/{eye}/{vid}")
print("─" * 70)

# Recompute whole
from features.density import whole_image_density_features as _whole, zone_density_features as _zonef
recomputed = {}
recomputed.update(_whole(vs_m, av_m, dc_m, valid_mask=whole_valid))
recomputed.update(_zonef(vs_m, av_m, dc_m, zones, valid_mask=common_valid))

# Compare with production row (every key that overlaps)
diff_rows = []
for k, v_mine in recomputed.items():
    if k not in prod_row.index:
        continue
    v_prod = prod_row[k]
    try:
        v_mine_f = float(v_mine); v_prod_f = float(v_prod)
    except (TypeError, ValueError):
        continue
    delta = v_mine_f - v_prod_f
    diff_rows.append({
        "feature": k,
        "verify":  round(v_mine_f, 6),
        "production": round(v_prod_f, 6),
        "delta": delta,
        "match": "✓" if abs(delta) < 1e-6 else "✗",
    })

cmp_df = pd.DataFrame(diff_rows).sort_values("feature")
n_match = (cmp_df["match"] == "✓").sum()
n_total = len(cmp_df)
print(f"\n{n_match}/{n_total} density features match (|delta| < 1e-6)")

if (cmp_df["match"] == "✗").any():
    print(f"\nMISMATCHES:")
    print(cmp_df[cmp_df["match"] == "✗"].to_string(index=False))
else:
    print(f"\nAll density features match ✓  —  pipeline is reproducible.")

# Show the first 15 for inspection
print(f"\nSample (first 15):")
print(cmp_df.head(15).to_string(index=False))
